# 🎓 MY-AI — বাংলা + English ট্রেনিং (Google Colab, ১–৫ ঘণ্টা)

এই নোটবুকটা একটা ওপেন-সোর্স মডেলকে **আপনার নিজের AI** বানায় — বাংলা, ইংরেজি আর বাংলিশ তিনটাতেই কথা বলতে শেখে।

> ⏱️ **আপনি শুধু ঘণ্টা বলে দেবেন (১–৫), বাকিটা নোটবুক নিজেই হিসাব করবে।**
> স্ক্রিপ্ট আপনার GPU-র আসল স্পিড মেপে ঠিক করে কত স্টেপ চালানো যাবে, তারপর
> সময় শেষ হওয়ার আগেই ট্রেনিং থামিয়ে মডেল সেভ করে ফেলে। কখনো "অর্ধেক ট্রেনিং, কিছুই সেভ হয়নি" হবে না।

| প্লan | সময় | মডেল | কী শিখবে |
|---|---|---|---|
| ⚡ Quick | ~১ ঘণ্টা | Qwen2.5-1.5B | দৈনন্দিন কথাবার্তা, সহজ প্রশ্নোত্তর |
| ⭐ Balanced (সুপারিশ) | ~৩ ঘণ্টা | Qwen2.5-1.5B | কথাবার্তা + লেখা + অনুবাদ + কিছুটা রিজনিং |
| 🚀 Advanced | ~৫ ঘণ্টা | Qwen2.5-3B | লম্বা/গোছানো উত্তর, রিজনিং, কোড |

### শুরুর আগে (একবারই)
1. উপরে **Runtime → Change runtime type → T4 GPU** সিলেক্ট করে **Save** করুন
2. সেলগুলো **উপর থেকে নিচে** একটা একটা করে চালান

> 💡 ফ্রি Colab ~৩-৪ ঘণ্টা পর ডিসকানেক্ট করে দিতে পারে। সেল ২-এ Google Drive যুক্ত করলে
> চেকপয়েন্ট Drive-এ সেভ হবে আর সেল ৭ দিয়ে ঠিক যেখানে থেমেছিল সেখান থেকেই আবার শুরু করা যাবে।


---
## ধাপ ১ — GPU চেক


In [ ]:
# GPU আছে কিনা দেখুন — 'Tesla T4' / 'L4' / 'A100' দেখালে OK
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), '❌ GPU নেই! Runtime → Change runtime type → T4 GPU দিন, তারপর আবার চালান।'


---
## ধাপ ২ — Google Drive যুক্ত করুন (সুপারিশ)

Drive যুক্ত করলে ট্রেনিং-চেকপয়েন্ট আর ফাইনাল মডেল Drive-এ থাকবে — Colab ডিসকানেক্ট হলেও কিছু হারাবে না।
না চাইলে এই সেলটা স্কিপ করতে পারেন (তখন সব Colab-এর ভেতরেই থাকবে)।


In [ ]:
USE_DRIVE = True  # False করলে Drive লাগবে না

OUTPUT_DIR = '/content/my-ai-model'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/my-ai-model'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('📁 মডেল এখানে সেভ হবে:', OUTPUT_DIR)


---
## ধাপ ৩ — ট্রেনিং স্ক্রিপ্ট আনুন

রিপো থেকে সরাসরি ক্লোন হবে — কিছু আপলোড করার দরকার নেই।


In [ ]:
REPO = 'https://github.com/tufazzal513/NO-RULES-AI.git'
BRANCH = 'main'   # অন্য ব্রাঞ্চ হলে এখানে বদলান

import os, shutil
if os.path.exists('/content/NO-RULES-AI'):
    shutil.rmtree('/content/NO-RULES-AI')
!git clone -q --depth 1 -b $BRANCH $REPO /content/NO-RULES-AI
%cd /content/NO-RULES-AI/training
!ls -1
print('\n✅ স্ক্রিপ্ট রেডি')


---
## ধাপ ৪ — প্যাকেজ ইনস্টল (৩–৫ মিনিট, একবারই)


In [ ]:
%%capture
# Unsloth = ২× দ্রুত ট্রেনিং, ৭০% কম VRAM
!pip install -q -U unsloth
!pip install -q -U "trl>=0.9" "transformers>=4.44" datasets accelerate peft bitsandbytes


In [ ]:
import unsloth, trl, transformers, datasets
print('unsloth', unsloth.__version__, '| trl', trl.__version__, '| transformers', transformers.__version__)
print('✅ ইনস্টল শেষ')


---
## ধাপ ৫ — (ঐচ্ছিক) নিজের ডাটা আপলোড করুন

MY-AI ড্যাশবোর্ড → **Datasets → Export** থেকে নামানো `myai-dataset.jsonl` আপলোড করলে
মডেল **আপনার নিজের ঢঙে** কথা বলা শিখবে। না থাকলে সেলটা স্কিপ করুন — রিপোর ভেতরের
বাংলা-ইংরেজি-বাংলিশ সিড ডাটা এমনিতেই ব্যবহার হবে।


In [ ]:
from google.colab import files
print('myai-dataset.jsonl বাছাই করুন (না থাকলে Cancel চাপুন)…')
try:
    uploaded = files.upload()
    print('✅ আপলোড:', list(uploaded.keys()))
except Exception as e:
    print('স্কিপ করা হলো:', e)


---
## ধাপ ৬ — ⚙️ সেটিংস (এখানেই সব ঠিক করুন)


In [ ]:
#@title আপনার ট্রেনিং প্ল্যান { display-mode: 'form' }

PLAN = 'Balanced (~3h)'  #@param ['Quick (~1h)', 'Balanced (~3h)', 'Advanced (~5h)']
ASSISTANT_NAME = 'MY-AI'  #@param {type:'string'}
OWNER_NAME = ''  #@param {type:'string'}
EXPORT_GGUF = True  #@param {type:'boolean'}

PLANS = {
    'Quick (~1h)':    dict(hours=1.0, recipe='basic',    model='unsloth/Qwen2.5-1.5B-Instruct', seq=1024),
    'Balanced (~3h)': dict(hours=3.0, recipe='balanced', model='unsloth/Qwen2.5-1.5B-Instruct', seq=1024),
    'Advanced (~5h)': dict(hours=5.0, recipe='advanced', model='unsloth/Qwen2.5-3B-Instruct',   seq=1024),
}
P = PLANS[PLAN]

print(f"⏱️  সময়     : {P['hours']} ঘণ্টা")
print(f"🍳 রেসিপি   : {P['recipe']}")
print(f"🧠 মডেল     : {P['model']}")
print(f"📁 আউটপুট  : {OUTPUT_DIR}")
print(f"🤖 নাম      : {ASSISTANT_NAME}")


---
## ধাপ ৭ — 🚀 ট্রেনিং শুরু

এই একটা সেলেই সব হবে:

1. **ডাটা** — Hugging Face থেকে বাংলা (Bangla-Instruct, Bangla-Alpaca-Orca …) + ইংরেজি (UltraChat, Alpaca …) ডাটা নামবে, আপনার নিজের ডাটার সাথে মিশবে, ভাষা অনুযায়ী ব্যালান্স হবে
2. **স্পিড প্রোব** — কয়েকটা স্টেপ চালিয়ে আপনার GPU কত দ্রুত তা মাপা হবে
3. **ট্রেনিং** — বাজেটের ভেতরে যত স্টেপ সম্ভব, cosine schedule সহ
4. **সেভ + GGUF** — Ollama-তে চালানোর ফাইল
5. **টেস্ট** — বাংলা/ইংরেজি/বাংলিশ প্রশ্ন করে উত্তর দেখাবে

> ⏳ সেলটা চলাকালীন ট্যাব খোলা রাখুন। মাঝে মাঝে `⏳ step 120/900 — 96 min left` এমন লাইন দেখবেন।
> **ডিসকানেক্ট হলে?** নিচের সেলে `--resume` যোগ করে আবার চালান — চেকপয়েন্ট থেকে শুরু হবে।


In [ ]:
import shlex, subprocess, sys

cmd = [
    'python', 'train_lora.py',
    '--model', P['model'],
    '--recipe', P['recipe'],
    '--time-budget-hours', str(P['hours']),
    '--max-seq-length', str(P['seq']),
    '--assistant-name', ASSISTANT_NAME,
    '--owner-name', OWNER_NAME,
    # নিজের ডাটা — ফাইল না থাকলে অটো স্কিপ হবে, ক্র্যাশ করবে না
    '--extra-jsonl', 'myai-dataset.jsonl',
    '--extra-jsonl', '/content/myai-dataset.jsonl',
    '--extra-jsonl', '../data/import/bangla-english-banglish-chat.jsonl',
    '--output', OUTPUT_DIR,
]
if EXPORT_GGUF:
    cmd.append('--export-gguf')
# ডিসকানেক্টের পর আবার চালালে নিচের লাইনের # সরান:
# cmd.append('--resume')

print('$', ' '.join(shlex.quote(c) for c in cmd), '\n', flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\n=== exit code:', proc.returncode, '===')


---
## ধাপ ৮ — 💬 নিজে চ্যাট করে দেখুন

ট্রেনিং শেষ হওয়ার পর এই সেলটা চালিয়ে সরাসরি প্রশ্ন করুন (বাংলা / English / Banglish)।


In [ ]:
import os, sys, subprocess

# --- রানটাইম রিস্টার্ট/নতুন সেশনে /content মুছে যায় ---
# train_lora.py না পেলে রিপো নিজে থেকেই আবার ক্লোন করা হবে:
TRAIN_DIR = '/content/NO-RULES-AI/training'
if not os.path.isfile(os.path.join(TRAIN_DIR, 'train_lora.py')):
    print('⚠️ train_lora.py পাওয়া যায়নি — রিপো আবার ক্লোন করা হচ্ছে…')
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', 'main',
                    'https://github.com/tufazzal513/NO-RULES-AI.git', '/content/NO-RULES-AI'], check=True)
    print('✅ ক্লোন শেষ')
sys.path.insert(0, TRAIN_DIR)

# --- রানটাইম রিস্টার্টে Unsloth ইনস্টলও মুছে যায় ---
try:
    from unsloth import FastLanguageModel
except ModuleNotFoundError:
    print('⚠️ Unsloth ইনস্টল নেই — বসানো হচ্ছে (কয়েক মিনিট)…')
    subprocess.run(['pip', 'install', '-q', '-U', 'unsloth'], check=True)
    subprocess.run(['pip', 'install', '-q', '-U', 'trl>=0.9', 'transformers>=4.44',
                    'datasets', 'accelerate', 'peft', 'bitsandbytes'], check=True)
    from unsloth import FastLanguageModel

import torch
from train_lora import SYSTEM_PROMPT


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=OUTPUT_DIR, max_seq_length=1024, dtype=None, load_in_4bit=True)
FastLanguageModel.for_inference(model)

def ask(question, max_new_tokens=300):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.7,
                         top_p=0.9, repetition_penalty=1.1, do_sample=True,
                         pad_token_id=tokenizer.eos_token_id)
    reply = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print('👤', question)
    print('🤖', reply.strip(), '\n')

ask('তুমি কে? সংক্ষেপে বলো।')
ask('বাংলাদেশ সম্পর্কে ৩ লাইনে লেখো।')
ask('Write a short professional email asking for one day of leave.')
ask('amar mon kharap, ki korte pari?')


In [ ]:
# নিজের প্রশ্ন এখানে লিখুন
ask('তোমার প্রশ্ন এখানে লিখুন')


---
## ধাপ ৯ — ⬇️ মডেল ডাউনলোড

Drive ব্যবহার করলে মডেল এমনিতেই Drive-এ আছে — এই সেলটা তখন ঐচ্ছিক।


In [ ]:
import glob, os
print('📦 ফাইল:')
for f in sorted(glob.glob(OUTPUT_DIR + '/*')):
    print(f'  {os.path.basename(f):<40} {os.path.getsize(f) / 1e6:>8.1f} MB' if os.path.isfile(f) else f'  {os.path.basename(f)}/')

# GGUF থাকলে শুধু সেটাই নামান (ছোট + Ollama-তে সরাসরি চলে)
gguf = glob.glob(OUTPUT_DIR + '/*.gguf')
from google.colab import files
if gguf:
    print('\n⬇️ GGUF নামছে:', gguf[0])
    files.download(gguf[0])
    files.download(OUTPUT_DIR + '/Modelfile')
else:
    !cd "{OUTPUT_DIR}" && zip -r -q /content/my-ai-model.zip .
    files.download('/content/my-ai-model.zip')


---
## ধাপ ১০ — নিজের PC-তে চালান (Ollama)

```bash
# ১. Ollama ইনস্টল করুন → https://ollama.com/download
# ২. .gguf আর Modelfile একই ফোল্ডারে রাখুন, তারপর:
ollama create my-ai -f Modelfile
ollama run my-ai
```

### MY-AI অ্যাপের সাথে যুক্ত করা
Ollama চালু থাকলে `http://localhost:11434/api/chat` এ POST করলেই উত্তর আসে —
বিস্তারিত: `training/GUIDE_COLAB_BN.md` ("অ্যাপের সাথে যুক্ত করা" অংশ)।

---
## ❓ সমস্যা হলে

| সমস্যা | সমাধান |
|---|---|
| `CUDA out of memory` | ধাপ ৭-এ `--batch-size 1 --grad-accum 16` যোগ করুন, বা ছোট মডেল (`Qwen2.5-0.5B-Instruct`) নিন |
| Colab ডিসকানেক্ট হয়ে গেছে | ধাপ ১→৪ আবার চালিয়ে ধাপ ৭-এ `--resume` যোগ করুন (Drive ব্যবহার করলে চেকপয়েন্ট আছে) |
| কোনো ডেটাসেট skip হয়েছে | সমস্যা নেই — বাকিগুলো দিয়েই মিক্স হয়। সবগুলো skip হলে ইন্টারনেট চেক করুন |
| GGUF এক্সপোর্ট ফেল | LoRA অ্যাডাপ্টার সেভ হয়েই আছে; `--export-gguf` ছাড়া আবার চালিয়ে পরে এক্সপোর্ট করুন |
| উত্তর বাংলা প্রশ্নে ইংরেজিতে আসে | ডাটা বাড়ান (`--rows 40000`) বা বেশি সময় দিন (`--time-budget-hours 5`) |

## 💡 সত্যি কথাটা

ফাইন-টিউনিং মডেলকে **ভাষা, ঢং আর উত্তরের ধরন** শেখায় — নতুন ফ্যাক্ট মুখস্থ করায় না।
নতুন তথ্য শেখাতে MY-AI-এর **Knowledge / RAG** ব্যবহার করুন।
সেরা ফল = **LoRA (ঢং) + RAG (তথ্য)** একসাথে।
